**Code statement:**   
This code follows only the same logic as the code provided by Matt Ryan et al. 

However, the code was written by me as my own work, which is quite different as Matt's code.

In [1]:
import pandas as pd

# Load dataset
df = pd.read_csv("../raw/australia.csv", na_values=[" ", "__NA__"], keep_default_na=True, low_memory=False)

# Create a missing value table
missing_value_df = pd.DataFrame([
    {
        "Variable Name": col,
        "Missing Value Count": df[col].isna().sum()
    }
    for col in df.columns
])

# Sort by missing value count and variable name
missing_value_df = missing_value_df.sort_values(
    by=["Missing Value Count", "Variable Name"], ascending=[True, True]
)

# Save the missing value table
missing_value_df.to_csv("../data/missing_value_counts.csv", index=False)

In [2]:
import pandas as pd

df = pd.read_csv("../raw/australia.csv", na_values=[" ", "__NA__"], keep_default_na=True,low_memory=False)

# Convert date variable
df["endtime"] = pd.to_datetime(df["endtime"].str.split().str[0], format="%d/%m/%Y")

# Drop columns with too many missing values
thresh_value = 11000
missing_value_df = pd.read_csv("../data/missing_value_counts.csv")
columns_to_drop = (
    missing_value_df
    .query("`Missing Value Count` > @thresh_value")["Variable Name"]
    .tolist()
)
df = df.drop(columns=columns_to_drop)

# Fill specific missing values
sdate = pd.to_datetime("2021-02-10")
edate = pd.to_datetime("2021-10-18")

mask = df["endtime"].between(sdate, edate)
phq4_cols = [f"PHQ4_{i}" for i in range(1, 5)]
d1_health_cols = (
    [f"d1_health_{i}" for i in range(1, 14)] +
    [f"d1_health_{i}" for i in range(98, 100)]
)
cols_to_fill = phq4_cols + d1_health_cols

# Fill missing value with N/A
df.loc[mask, cols_to_fill] = df.loc[mask, cols_to_fill].fillna("N/A")

# Remove remaining missing values
df = df.dropna()

# Convert into numeric values
risk_scale_map = {
    "7 - Agree": 7,
    "6": 6,
    "5": 5,
    "4": 4,
    "3": 3,
    "2": 2,
    "1 – Disagree": 1
}

for col in ["r1_1", "r1_2"]:
    df[col] = df[col].replace(risk_scale_map)

# Convert into numeric values
frequency_map = {
    "Always": 5,
    "Frequently": 4,
    "Sometimes": 3,
    "Rarely": 2,
    "Not at all": 1
}

i12_health_cols = [
    col for col in df.columns
    if col.startswith("i12_health_")
]

df[i12_health_cols] = df[i12_health_cols].replace(frequency_map)

# Create behaviour scales and binary variables
face_mask_cols = [
    "i12_health_1",
    "i12_health_22",
    "i12_health_23",
    "i12_health_25"
]

# Calculate face mask behaviour score
df["face_mask_behaviour_scale"] = df[face_mask_cols].median(axis=1)

# Convert into binary outcome
df["face_mask_behaviour_binary"] = df["face_mask_behaviour_scale"].map(
    lambda x: "Yes" if x >= 4 else "No"
)

protective_behaviour_cols = [
    col for col in df.columns
    if col.startswith("i12_")
]

# Calculate protective behaviour score
df["protective_behaviour_scale"] = df[protective_behaviour_cols].median(axis=1)

# Convert into binary outcome
df["protective_behaviour_binary"] = df["protective_behaviour_scale"].map(
    lambda x: "Yes" if x >= 4 else "No"
)

# Calculate protective behaviour score without mask
protective_behaviour_nomask_cols = [
    col for col in protective_behaviour_cols
    if col not in face_mask_cols
]
df["protective_behaviour_nomask_scale"] = df[protective_behaviour_nomask_cols].median(axis=1)

# Combine d1 health variables
d1_cols = [
    col for col in df.columns
    if col.startswith("d1_")
]

df["d1_comorbidities"] = "Yes"
df.loc[df["d1_health_99"] == "Yes", "d1_comorbidities"] = "No"
df.loc[df["d1_health_99"] == "N/A", "d1_comorbidities"] = "NA"
df.loc[df["d1_health_98"] == "Yes", "d1_comorbidities"] = "Prefer_not_to_say"
df = df.drop(columns=d1_cols)

# Week number
start_date = df["endtime"].min()
df["week_number"] = ((df["endtime"] - start_date).dt.days // 14) + 1

# Convert household size
household_map = {
    "1": 1,
    "2": 2,
    "3": 3,
    "4": 4,
    "5": 5,
    "6": 6,
    "7": 7,
    "8 or more": 8,
    "Prefer not to say": None,
    "Don't know": None
}
df["household_size"] = df["household_size"].map(household_map)
df = df.dropna()

# Drop variables not needed for modelling
cols_to_remove = ["qweek", "weight"] + protective_behaviour_cols
df = df.drop(columns=cols_to_remove)

# Save
df.to_csv("../data/cleaned_data.csv", index=False)

/var/folders/hx/9fvpdn8n3419511f3tf8p7tm0000gn/T/ipykernel_14608/1337990513.py:48: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[col] = df[col].replace(risk_scale_map)
/var/folders/hx/9fvpdn8n3419511f3tf8p7tm0000gn/T/ipykernel_14608/1337990513.py:64: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df[i12_health_cols] = df[i12_health_cols].replace(frequency_map)
